# Retail Sales Prediction: Modeling and Interpretation

This notebook loads a fresh copy of the raw dataset, performs pre-split cleaning only where safe, builds preprocessing pipelines, compares models, tunes a Random Forest, and interprets the final model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    BEST_MODEL_PATH,
    DATA_DICTIONARY,
    FIGURES_DIR,
    PROCESSED_DATA_PATH,
    RANDOM_STATE,
    RAW_DATA_PATH,
    TARGET,
)
from src.data_loader import (
    basic_cleaning,
    load_raw_data,
    missing_value_report,
    restore_placeholders_to_null,
    save_processed_snapshot,
    summarize_numeric_columns,
)
from src.evaluation import evaluate_regression_model
from src.modeling import (
    build_linear_regression_pipeline,
    build_random_forest_pipeline,
    make_train_test_split,
    split_features_target,
    tune_random_forest,
)
from src.preprocessing import make_preprocessor
from src.visualization import (
    plot_correlation_heatmap,
    plot_linear_regression_coefficients,
    plot_model_comparison,
    plot_outlet_type_vs_sales,
    plot_rf_feature_importance,
    save_figure,
    set_plot_style,
)

pd.set_option("display.max_columns", 100)
set_plot_style()

## Fresh Data Load

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
print(raw_df.shape)
display(raw_df.head())

## Pre-Split Cleaning

To avoid data leakage, only duplicate removal and category standardization happen before splitting. Missing-value imputation is handled inside the scikit-learn preprocessing pipeline after the train-test split.

In [ ]:
model_df = basic_cleaning(raw_df, fill_placeholders=False)

print(f"Duplicate rows after cleaning: {model_df.duplicated().sum():,}")
print(sorted(model_df["Item_Fat_Content"].dropna().unique()))
display(missing_value_report(model_df))

## Feature and Target Split

In [ ]:
X, y = split_features_target(model_df)
X_train, X_test, y_train, y_test = make_train_test_split(X, y)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

## Preprocessing Object

In [ ]:
preprocessor = make_preprocessor(scale_numeric=True)
preprocessor

## Linear Regression

In [ ]:
linear_model = build_linear_regression_pipeline()
linear_model.fit(X_train, y_train)

linear_metrics = evaluate_regression_model(
    "Linear Regression", linear_model, X_train, y_train, X_test, y_test
)
display(linear_metrics.round(4))

Linear Regression has train R-squared 0.5616 and test R-squared 0.5671. The two scores are close, so this model is not heavily overfit, but it underfits some of the nonlinear relationships in the data.

## Default Random Forest

In [ ]:
default_rf = build_random_forest_pipeline(n_estimators=200)
default_rf.fit(X_train, y_train)

rf_metrics = evaluate_regression_model(
    "Random Forest", default_rf, X_train, y_train, X_test, y_test
)
display(rf_metrics.round(4))

The default Random Forest has train R-squared 0.9395 and test R-squared 0.5588. This large gap indicates overfitting. Its test score is also lower than Linear Regression, so tuning is needed before recommending a tree-based model.

## Tuned Random Forest

In [ ]:
search = tune_random_forest(build_random_forest_pipeline(), X_train, y_train)
print("Best parameters:", search.best_params_)
print(f"Best cross-validation R2: {search.best_score_:.4f}")

best_params = {key.replace("model__", ""): value for key, value in search.best_params_.items()}
tuned_rf = build_random_forest_pipeline(**best_params)
tuned_rf.fit(X_train, y_train)

tuned_metrics = evaluate_regression_model(
    "Tuned Random Forest", tuned_rf, X_train, y_train, X_test, y_test
)
display(tuned_metrics.round(4))

In [ ]:
metrics_df = pd.concat([linear_metrics, rf_metrics, tuned_metrics], ignore_index=True)
metrics_df = metrics_df.round({"MAE": 2, "MSE": 2, "RMSE": 2, "R2": 4})
display(metrics_df)

metrics_df.to_csv(PROJECT_ROOT / "reports" / "model_metrics.csv", index=False)
plot_model_comparison(metrics_df, FIGURES_DIR / "model_comparison.png")
display(Image(filename=str(FIGURES_DIR / "model_comparison.png")))

The tuned Random Forest improved over the default Random Forest on the test set (0.6026 vs. 0.5588). The tuned model is the recommended model because it has the best test R-squared and the lowest RMSE among the compared models.

For a non-technical stakeholder: the tuned model explains about 60.3% of the variation in item sales on unseen data. I would also communicate RMSE ($1,047) because it describes the typical prediction error in the same sales units stakeholders understand.

## Linear Regression Coefficients

In [ ]:
coef_df = plot_linear_regression_coefficients(
    linear_model, FIGURES_DIR / "linreg_coefficients.png"
)
display(Image(filename=str(FIGURES_DIR / "linreg_coefficients.png")))
display(coef_df.head(10))

Top coefficient interpretation:

- `Outlet_Type_Grocery Store` has the strongest negative coefficient, meaning grocery store outlets are associated with much lower predicted sales than the baseline store type.
- `Item_MRP` has a strong positive coefficient, meaning higher-priced items tend to generate higher sales.
- `Outlet_Identifier_OUT027` has a strong positive coefficient, suggesting that this specific outlet performs above the baseline after preprocessing.

## Tree-Based Feature Importances

In [ ]:
importance_df = plot_rf_feature_importance(
    tuned_rf, FIGURES_DIR / "rf_feature_importance.png"
)
display(Image(filename=str(FIGURES_DIR / "rf_feature_importance.png")))
display(importance_df.head(10))

Top Random Forest feature interpretation:

- `Item_MRP` is the most important feature, showing that listed price is the strongest driver of predicted sales.
- `Outlet_Type_Grocery Store` is highly important, reinforcing that grocery outlets behave differently from supermarkets.
- `Outlet_Type_Supermarket Type3`, `Outlet_Identifier_OUT027`, and `Outlet_Establishment_Year` indicate that store format and store identity/history matter for sales prediction.

## Save Final Model

In [ ]:
import joblib

joblib.dump(tuned_rf, BEST_MODEL_PATH)
print(f"Saved final tuned model to: {BEST_MODEL_PATH}")

## Stakeholder Recommendation

I recommend using the tuned Random Forest for sales forecasting because it provides the strongest test-set performance while reducing the overfitting seen in the default forest. From a business perspective, the retailer should pay close attention to item price, store format, and high-performing outlet patterns when planning assortment, placement, and sales forecasts.